In [37]:
import fredapi
from dotenv import load_dotenv
import os
import pandas as pd
import sqlalchemy as db
from sqlalchemy import text

In [38]:
load_dotenv()
api_key = os.getenv("API_KEY")

In [39]:
fred = fredapi.Fred(api_key)

In [40]:
data = fred.get_series('IRLTLT01USM156N')

In [41]:
data_vintage = fred.get_series_vintage_dates('IRLTLT01USM156N')

In [42]:
pd.to_datetime(data_vintage)

DatetimeIndex(['2013-06-03', '2013-07-01', '2013-08-01', '2013-08-21',
               '2013-10-01', '2013-11-01', '2013-12-02', '2014-02-03',
               '2014-11-03', '2014-12-01',
               ...
               '2025-05-15', '2025-06-16', '2025-07-15', '2025-08-15',
               '2025-09-15', '2025-10-15', '2025-11-17', '2025-12-15',
               '2026-01-15', '2026-02-16'],
              dtype='datetime64[us]', length=135, freq=None)

In [43]:
data = data.dropna()

In [44]:
data.head()

1953-04-01    2.83
1953-05-01    3.05
1953-06-01    3.11
1953-07-01    2.93
1953-08-01    2.95
dtype: float64

In [45]:
data.tail()

2025-09-01    4.12
2025-10-01    4.06
2025-11-01    4.09
2025-12-01    4.14
2026-01-01    4.21
dtype: float64

In [46]:
data.describe

<bound method NDFrame.describe of 1953-04-01    2.83
1953-05-01    3.05
1953-06-01    3.11
1953-07-01    2.93
1953-08-01    2.95
              ... 
2025-09-01    4.12
2025-10-01    4.06
2025-11-01    4.09
2025-12-01    4.14
2026-01-01    4.21
Length: 874, dtype: float64>

In [47]:
type(data)

pandas.Series

In [48]:
print(fred.get_series_all_releases('IRLTLT01USM156N'))

           realtime_start                 date value
0     2024-04-10 00:00:00  1953-04-01 00:00:00  2.83
1     2024-04-10 00:00:00  1953-05-01 00:00:00  3.05
2     2024-04-10 00:00:00  1953-06-01 00:00:00  3.11
3     2024-04-10 00:00:00  1953-07-01 00:00:00  2.93
4     2024-04-10 00:00:00  1953-08-01 00:00:00  2.95
...                   ...                  ...   ...
2070  2025-10-15 00:00:00  2025-09-01 00:00:00  4.12
2071  2025-11-17 00:00:00  2025-10-01 00:00:00  4.06
2072  2025-12-15 00:00:00  2025-11-01 00:00:00  4.09
2073  2026-01-15 00:00:00  2025-12-01 00:00:00  4.14
2074  2026-02-16 00:00:00  2026-01-01 00:00:00  4.21

[2075 rows x 3 columns]


In [52]:
# Approximate fixed lags in days from value date to publication as FREDAPI doesn't offers real publish dates for old publications
rate_lags = {
    'UNRATE': 45,  # ~6 weeks after reference month end
    'CPIAUCSL': 15,  # ~2 weeks after reference month end
    'M2SL': 30,
    'WALCL': 7,
    'NFCI': 7,
    'ICSA': 5,  # Thursdays, covers prior week
}

def get_fred_data(serie):
    '''
    Process and store in the db the indicator
    '''
    #get the data
    data = fred.get_series(serie)
    #drop nan values
    data = data.dropna()

    if serie in rate_lags:
        data.index = data.index + pd.DateOffset(days=rate_lags[serie])

    # convert Series to DataFrame
    data = data.reset_index()
    data.columns = ['date', 'value']

    # remove timezone info
    data['date'] = pd.to_datetime(data['date']).dt.tz_localize(None)

    # add series name
    data['serie'] = serie

    return data

get_fred_data('FEDFUNDS')

,date,value,serie
0,1954-07-01,0.80,FEDFUNDS
1,1954-08-01,1.22,FEDFUNDS
2,1954-09-01,1.07,FEDFUNDS
3,1954-10-01,0.85,FEDFUNDS
4,1954-11-01,0.83,FEDFUNDS
...,...,...,...
854,2025-09-01,4.22,FEDFUNDS
855,2025-10-01,4.09,FEDFUNDS
856,2025-11-01,3.88,FEDFUNDS
857,2025-12-01,3.72,FEDFUNDS


In [50]:
engine = db.create_engine('sqlite:///../data/data.db')
with engine.connect() as conn:
    conn.execute(text('ALTER TABLE Macro DROP COLUMN vintage_date'))
    conn.commit()

In [121]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd
from bs4 import BeautifulSoup
import time

In [123]:
driver = webdriver.Chrome()
driver.get("https://www.forexfactory.com/calendar")

#search the filter button
elements = driver.find_element(By.CLASS_NAME, "highlight.filters")
elements.click()
#deselect the countries
deselect_all = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div[2]/p/span/a[2]')
deselect_all.click()
#select only usa
usa_element = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div[2]/div/div[2]/div[5]/div[1]')
usa_element.click()
#apply changes
apply_changes = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[2]/div/table/tbody/tr/td[3]/input[1]')
apply_changes.click()
time.sleep(5)

#search te calendar
calendar_element = driver.find_element(By.CLASS_NAME, 'calendar__options.left')
calendar_element.click()
#select date range
date_range_element = driver.find_element(By.ID, 'calendar-date-range-1')
date_range_element.click()
date_range_element.clear()
date_range_element.send_keys('Jan 1, 2007 – Jan 31, 2007')
#apply date settings
apply_date = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[1]/div/table/tbody/tr/td[2]/input[1]')
apply_date.click()
time.sleep(3)

#scroll page
actions = ActionChains(driver)
actions.send_keys(Keys.END).perform()
time.sleep(3)

#get html
page_html = driver.page_source
#read the html with beautifulsoup
soup = BeautifulSoup(page_html, 'lxml')

#get the event table
tables = soup.find_all('table', {'class': 'calendar__table'})
rows = soup.find_all('tr', {'class': 'calendar__row'})

day = ""

for row in rows:
    # Get the date and the event name
    date_span = row.find('span', {'class': 'date'})
    event_span = row.find('span', {'class': 'calendar__event-title'})

    if date_span is not None:
        # stores the data for the same event days
        day = date_span.text.split()[-1]

    if event_span is not None:
        event = event_span.text.strip()
        if event:
            print(f"{day} - {event}")


1 - Bank Holiday
3 - ADP Non-Farm Employment Change
3 - ISM Manufacturing PMI
3 - Construction Spending m/m
3 - ISM Manufacturing Prices
3 - Total Vehicle Sales
3 - FOMC Meeting Minutes
4 - Challenger Job Cuts y/y
4 - Unemployment Claims
4 - ISM Services PMI
4 - Pending Home Sales m/m
4 - Factory Orders m/m
4 - Crude Oil Inventories
5 - Non-Farm Employment Change
5 - Unemployment Rate
5 - Average Hourly Earnings m/m
5 - Natural Gas Storage
5 - Fed Chairman Bernanke Speaks
8 - FOMC Member Kohn Speaks
8 - Consumer Credit m/m
9 - NFIB Small Business Index
9 - RCM/TIPP Economic Optimism
10 - Trade Balance
10 - Final Wholesale Inventories m/m
10 - JOLTS Job Openings
10 - Crude Oil Inventories
11 - FOMC Member Geithner Speaks
11 - Unemployment Claims
11 - Natural Gas Storage
12 - Core Retail Sales m/m
12 - Retail Sales m/m
12 - Import Prices m/m
12 - Business Inventories m/m
12 - Federal Budget Balance
15 - Bank Holiday
16 - Empire State Manufacturing Index
17 - PPI m/m
17 - Core PPI m/m
17 

In [106]:
print(rows)

[<tr class="calendar__header--desktop subhead"><th class="calendar__date">Date</th> <th class="calendar__time"><a href="/timezone" title="Time Options">4:19pm</a></th> <th class="calendar__currency">Currency</th> <th class="calendar__impact">Impact</th> <th class="calendar__event"></th> <!-- --> <th class="calendar__sub">Alerts</th> <th class="calendar__detail">Detail</th> <th class="calendar__actual">Actual</th> <th class="calendar__forecast">Forecast</th> <th class="calendar__previous">Previous</th> <th class="calendar__graph">Graph</th></tr>, <tr class="calendar__header--mobile subhead"><th colspan="4"><div class="calendar__header-time"><a href="/timezone" title="Time Options">4:19pm</a> <span class="icon icon--loader" style="display: none;"></span></div></th> <th>Actual</th></tr>, <tr class="calendar__borderfix borderfix"><td></td></tr>, <tr class="calendar__row calendar__row--day-breaker"><td class="calendar__cell" colspan="10">Mon <span>Jan 1</span></td></tr>, <tr class="calendar

In [67]:
tb = pd.read_html(rows)
tb

TypeError: cannot parse from 'ResultSet'